# 第2章：表格、缺失与尺度

**学生实验｜全书深化稿**

## 背景与数据
葡萄酒化学分析：178条、13项测量、三个来源类别；缺失只在副本上人为加入。

## 本次要回答的问题
核对行身份、训练中位数和标准化。追踪哪一步学习了数据参数。

先写下预判，再运行，最后用实际结果解释；不需要结果符合预期才算成功。

In [1]:
from pathlib import Path
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "experiments/common.py").exists()
             and (p / "main.tex").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the extracted textbook project.")
sys.path.insert(0, str(root / "experiments"))
print("Project:", root)


Project: /mnt/data/book_v3


## 可检查的完整实验代码

下方与对应`experiments`脚本一致。共享的`common.py`只提供固定数据划分、保存表图和绘图设置；模型、目标、参数及比较规则在此明确写出。阅读函数后，再运行下一单元执行实验。

In [2]:
"""Chapter 2: missing values, train-only transforms, and column integrity."""
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from common import *

def run():
    Xtr,Xva,ytr,yva,ds=development_trainval('wine')
    original=Xtr.copy(); modified=Xtr.copy()
    rng=np.random.default_rng(12)
    positions=rng.choice(len(modified),size=8,replace=False)
    modified.iloc[positions,0]=np.nan
    pipe=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
    ztr=pipe.fit_transform(modified);zva=pipe.transform(Xva)
    assert np.isfinite(ztr).all() and np.isfinite(zva).all()
    assert np.allclose(ztr.mean(axis=0),0,atol=1e-12)
    rows=[]
    for j in [0,4,12]:
        rows.append({'feature':Xtr.columns[j], 'training_mean_raw':float(original.iloc[:,j].mean()),
        'training_std_ddof0':float(original.iloc[:,j].std(ddof=0)),
        'training_mean_scaled':float(ztr[:,j].mean()),'validation_mean_scaled':float(zva[:,j].mean())})
    df=table('ch02_scaling',rows);show_result(df)
    table('ch02_missing_positions',[{'training_position':int(i),'original_row':int(Xtr.index[i]),'field':Xtr.columns[0]} for i in positions])
    # A row permutation is harmless only if X and y are permuted together.
    toy=pd.DataFrame({'id':['A','B','C'],'x':[1.,2.,3.]})
    labels=pd.DataFrame({'id':['C','A','B'],'target':[30.,10.,20.]})
    merged=toy.merge(labels,on='id',validate='one_to_one')
    assert np.array_equal(merged.target,[10,20,30])
    table('ch02_safe_join',merged)

In [3]:
result = run()
if result is not None:
    print(result)

  feature  training_mean_raw  training_std_ddof0  training_mean_scaled  validation_mean_scaled
  alcohol          12.934057            0.811024         -8.188222e-16                0.211713
magnesium          98.877358           14.046891         -7.122185e-17                0.212414
  proline         735.547170          298.048215          1.120697e-16                0.052033


## 检查结果文件与图

下面显示该实验保存的图；所有坐标、样本与数值条件应与正文一致。数据表保存在`results/`。修改代码后，请另存结果或注明变化，不把新配置的输出与正文旧数值混用。

In [4]:
from IPython.display import display, Image
files = sorted((root / "figures").glob("v3_ch02_*.png"))
for path in files:
    print(path.name)
    display(Image(filename=str(path), width=720))
if not files:
    print("This experiment uses tables rather than a new figure.")

This experiment uses tables rather than a new figure.


## 运行后的分析

解释一个与你的预判一致的结果，再解释一个需要进一步检查的现象。指出训练、验证和测试各使用哪些记录。

核对行身份、训练中位数和标准化。追踪哪一步学习了数据参数。

**提交要求：**保存可顺序运行的Notebook、一个关键结果表、必要图形，以及150—300字的机制与限制解释。完成正文D题中与本实验相关的计算。不得仅提交最高分截图。